# Module 5 Exercise (Solution): Pretraining objectives side-by-side — BERT, GPT-2, T5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/05-llm-lineage/exercise_solution.ipynb)

Module page: [Module 5: The LLM Lineage](https://nsteve2407.github.io/llm-transformers-course/modules/05-llm-lineage/)

This is the first notebook in the course to use the `transformers` library. We load three real pretrained
architectures (`bert-base-uncased`, `gpt2`, `t5-small`) and, for each, implement **from scratch** (not via a
library collator) the masking/training-setup logic that defines its pretraining objective:

**Part 1**: BERT's masked-language-modeling (MLM) collator, implementing the 80/10/10 rule token by token.

**Part 2**: GPT-2's causal-LM setup (shift-by-one labels, generation), plus a hands-on demonstration of
in-context learning (ICL) via hand-written 0/1/5-shot prompts for a toy sentiment-classification task.

**Part 3**: T5's span-corruption collator -- contiguous spans (not independent tokens) replaced by sentinel
tokens, with the encoder/decoder input-target pairing T5 was pretrained on.

**Part 4**: a side-by-side attention-pattern visualization confirming BERT is bidirectional and GPT-2 is
causal.

`transformers`' own collators (`DataCollatorForLanguageModeling`, etc.) and model classes (`BertForMaskedLM`,
`GPT2LMHeadModel`, `T5ForConditionalGeneration`) are used *after* the from-scratch masking/collation logic --
never to replace it -- exactly analogous to how Module 4 used `nn.MultiheadAttention` only as a correctness
oracle, not inside the from-scratch implementation.

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers

In [ ]:
import os
import math
import random
import time

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Design choices and judgment calls (documented up front)

- **`SMOKE_TEST` model-loading pattern**: `load_model(name, model_cls)` below picks
  `model_cls.from_config(AutoConfig.from_pretrained(name))` (downloads only the small config JSON, builds a
  randomly-initialized model with the right shapes) when `SMOKE_TEST=1`, and `model_cls.from_pretrained(name)`
  (full pretrained weights, hundreds of MB) otherwise. Tokenizers are small (vocab/merges files, KB-scale) and
  always load via `from_pretrained` in both branches. This means: under `SMOKE_TEST`, every *shape* and
  *mechanism* check in this notebook (masking logic, tensor shapes, causal-vs-bidirectional attention
  structure, code paths running without error) is fully exercised, but anything that depends on *learned*
  weights (masked-token accuracy being meaningfully above chance, in-context-learning accuracy improving with
  more shots, T5 producing sensible span fills) is not meaningful with random weights -- those cells are
  explicitly gated on `SMOKE_TEST` and skipped with a printed note rather than silently producing misleading
  numbers.
- **`BertForMaskedLM`/`GPT2LMHeadModel`/`T5ForConditionalGeneration` (via the `AutoModelFor...` classes) are
  used directly** rather than manually attaching an LM head to each base model. The exercise is about
  implementing the *masking/collation logic* that defines each objective, not re-deriving "a linear layer
  tied to the embedding matrix" three times -- `transformers` already provides the correctly-tied head for
  each architecture.
- **Toy, hand-written data throughout** (a handful of sentences, a small sentiment-classification pool) --
  no dataset downloads. This keeps every cell fast regardless of `SMOKE_TEST` and keeps the notebook's focus
  on the collation/objective logic rather than data plumbing.
- **`mlm_probability=0.15`** (Part 1) and **`noise_density=0.15`, `mean_span_length=3`** (Part 3) match the
  original BERT and T5 papers' defaults.
- **Few-shot accuracy is measured via next-token logit comparison** (`" positive"` vs. `" negative"`, both
  single GPT-2 BPE tokens) rather than generating free text and parsing it -- deterministic, and avoids
  confounding "did the model learn the task" with "did greedy/sampled generation happen to produce a
  parseable string."
- **Models stay on CPU** (`load_model` does not call `.to(device)`, even though `device` is detected above).
  Every forward/generate call here is a handful of short toy sequences, not a training loop, so CPU is fast
  enough, and it avoids threading `.to(device)` through every tensor the from-scratch collator functions
  create.

In [ ]:
def load_model(name, model_cls):
    '''SMOKE_TEST=1: build model_cls from a downloaded config only (random weights, correct shapes,
    tiny download). SMOKE_TEST unset: full model_cls.from_pretrained(name) (real weights).

    Deliberately kept on CPU (not moved to `device`) -- every forward/generate call in this notebook is a
    handful of short toy sequences (no training loop), so CPU is plenty fast, and it sidesteps having to
    thread .to(device) through every tensor created by the from-scratch collator functions below.
    '''
    if SMOKE_TEST:
        config = AutoConfig.from_pretrained(name)
        model = model_cls.from_config(config)
    else:
        model = model_cls.from_pretrained(name)
    return model.eval()

## Setup: load tokenizers and models

Three architectures, three objectives: `bert-base-uncased` (encoder-only, MLM), `gpt2` (decoder-only, causal
LM), `t5-small` (encoder-decoder, span corruption). We time the loads so the `SMOKE_TEST` timing claim in
Part 0's docstring is directly checkable in this notebook's own output.

In [ ]:
t0 = time.time()

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = load_model("bert-base-uncased", AutoModelForMaskedLM)

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token  # GPT-2 has no pad token by default
gpt2_model = load_model("gpt2", AutoModelForCausalLM)

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_model = load_model("t5-small", AutoModelForSeq2SeqLM)

elapsed = time.time() - t0
print(f"Loaded 3 tokenizers + 3 models in {elapsed:.1f}s (SMOKE_TEST={SMOKE_TEST})")
for tag, model in [("bert", bert_model), ("gpt2", gpt2_model), ("t5", t5_model)]:
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {tag}: {n_params:,} parameters")
if SMOKE_TEST:
    # A full download of all three models' pretrained weights (~440MB + ~550MB + ~240MB) would take
    # noticeably longer than this config-only path -- see the notebook's smoke-test report for a
    # from_pretrained timing comparison run separately.
    print("SMOKE_TEST=1: models above have RANDOM weights (config-only load, no weight download).")

## Part 1: BERT's MLM collator -- the 80/10/10 rule, from scratch

For each token independently selected into the 15% "masked" set:
- **80%** of the time, replace it with `[MASK]`.
- **10%** of the time, replace it with a random vocabulary token.
- **10%** of the time, leave it unchanged.

In all three cases the label at that position is the *original* token id (so the model must predict the
true token from context alone), and every *non-selected* position gets label `-100`, which is
`CrossEntropyLoss`'s default `ignore_index` -- only the 15% masked positions contribute to the loss.

Why not just always insert `[MASK]`? Because `[MASK]` never appears at fine-tuning time, so a model that
learns "only predict something meaningful when you see `[MASK]`" would have a train/inference mismatch. The
10% random-token and 10% unchanged cases force the model to build a genuine contextual representation for
*every* input token, not just the ones it knows are masked.

We explicitly avoid masking BERT's special tokens (`[CLS]`, `[SEP]`, `[PAD]`) via
`tokenizer.get_special_tokens_mask`.

In [ ]:
def mask_tokens_mlm(input_ids, tokenizer, mlm_probability=0.15):
    '''input_ids: (batch, seq_len) LongTensor of token ids (already padded).

    Returns (masked_input_ids, labels):
      - masked_input_ids: input_ids with ~mlm_probability of (non-special) positions modified per the
        80/10/10 rule (80% -> [MASK], 10% -> random vocab token, 10% -> unchanged).
      - labels: input_ids.clone() with -100 at every position that was NOT selected for masking, and the
        ORIGINAL token id at every position that WAS selected (regardless of which of the 80/10/10 branches
        it took) -- this is what -100-masked labels means: only masked positions contribute to the loss.
    '''
    labels = input_ids.clone()

    # Select ~mlm_probability of tokens, but never special tokens ([CLS]/[SEP]/[PAD]/...).
    probability_matrix = torch.full(labels.shape, mlm_probability)
    special_tokens_mask = [
        tokenizer.get_special_tokens_mask(seq, already_has_special_tokens=True) for seq in labels.tolist()
    ]
    special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
    probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
    masked_indices = torch.bernoulli(probability_matrix).bool()

    # Only masked positions contribute to the loss; everything else is ignored via -100.
    labels[~masked_indices] = -100

    masked_input_ids = input_ids.clone()

    # 80% of the masked positions -> [MASK].
    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    masked_input_ids[indices_replaced] = tokenizer.convert_tokens_to_ids(tokenizer.mask_token)

    # Of the remaining 20% of masked positions, half (i.e. 10% of all masked positions) -> a random token.
    indices_random = (
        torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
    )
    random_words = torch.randint(len(tokenizer), labels.shape, dtype=torch.long)
    masked_input_ids[indices_random] = random_words[indices_random]

    # The remaining 10% of masked positions are left unchanged -- masked_input_ids already equals
    # input_ids there since we started from a clone and only wrote into indices_replaced/indices_random.
    return masked_input_ids, labels

In [ ]:
mlm_sentences = [
    "the quick brown fox jumps over the lazy dog",
    "transformers have become the dominant architecture in natural language processing",
    "attention is all you need according to the famous paper title",
    "bert uses masked language modeling for pretraining",
] * 8  # repeated so the 80/10/10 split is visible at a reasonable sample size

mlm_batch = bert_tokenizer(mlm_sentences, padding=True, truncation=True, return_tensors="pt")
masked_input_ids, mlm_labels = mask_tokens_mlm(mlm_batch["input_ids"], bert_tokenizer)

masked_positions = mlm_labels != -100
n_total = mlm_batch["input_ids"].numel()
n_masked = masked_positions.sum().item()
still_original = (masked_input_ids[masked_positions] == mlm_labels[masked_positions]).sum().item()
became_mask_tok = (masked_input_ids[masked_positions] == bert_tokenizer.mask_token_id).sum().item()
became_random = n_masked - still_original - became_mask_tok

print(f"total tokens: {n_total}, masked (selected): {n_masked} ({100 * n_masked / n_total:.1f}%)")
print(f"  -> [MASK]:        {became_mask_tok} ({100 * became_mask_tok / n_masked:.1f}%, target 80%)")
print(f"  -> random token:  {became_random} ({100 * became_random / n_masked:.1f}%, target 10%)")
print(f"  -> unchanged:     {still_original} ({100 * still_original / n_masked:.1f}%, target 10%)")
print("(sampling noise at this batch size -- ratios are close to, not exactly, 80/10/10)")

print()
print("original: ", bert_tokenizer.decode(mlm_batch["input_ids"][0], skip_special_tokens=False))
print("masked:   ", bert_tokenizer.decode(masked_input_ids[0], skip_special_tokens=False))

In [ ]:
with torch.no_grad():
    mlm_out = bert_model(
        input_ids=masked_input_ids, attention_mask=mlm_batch["attention_mask"], labels=mlm_labels
    )

mlm_preds = mlm_out.logits.argmax(dim=-1)
mlm_top1_acc = (mlm_preds[masked_positions] == mlm_labels[masked_positions]).float().mean().item()
print(f"MLM loss: {mlm_out.loss.item():.3f}")
print(f"Masked-token top-1 accuracy: {mlm_top1_acc:.3f}")

if SMOKE_TEST:
    print(
        "SMOKE_TEST=1: bert_model has random weights, so top-1 accuracy near chance "
        f"(~1/{len(bert_tokenizer)} = {1 / len(bert_tokenizer):.2e}) is expected, not a bug."
    )
else:
    print("Real pretrained bert-base-uncased weights: accuracy here reflects genuine MLM competence.")

## Part 2: GPT-2's causal-LM setup and in-context learning

### Causal-LM training/inference setup

GPT-2 is trained with a single objective: predict token `t+1` from tokens `<=t`. Concretely, given
`input_ids` of length `L`, the labels are `input_ids` shifted left by one position -- position `i`'s label
is `input_ids[i+1]` -- so the last position has no label (nothing follows it). `GPT2LMHeadModel` does this
shift internally when you pass `labels=input_ids` directly; below we replicate that shift manually and
confirm it produces the identical loss, to make the "shift by one" step concrete rather than a black box.

The **causal mask** itself is not something we implement here (unlike Module 4's from-scratch causal mask)
-- GPT-2's attention layers unconditionally apply a lower-triangular bias buffer internally, so position `i`
can never attend to position `j > i` regardless of what mask you pass in. Part 4 verifies this directly.

In [ ]:
causal_text = "the quick brown fox jumps over the lazy dog"
causal_ids = gpt2_tokenizer(causal_text, return_tensors="pt")["input_ids"]

with torch.no_grad():
    builtin_out = gpt2_model(input_ids=causal_ids, labels=causal_ids)
    logits_only = gpt2_model(input_ids=causal_ids).logits

# Manual replication of GPT2LMHeadModel's internal label-shift: predict token i+1 from logits at position i.
shift_logits = logits_only[:, :-1, :].contiguous()
shift_labels = causal_ids[:, 1:].contiguous()
manual_loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

print(f"built-in labels= loss:     {builtin_out.loss.item():.6f}")
print(f"manual shift-by-one loss:  {manual_loss.item():.6f}")
assert torch.allclose(builtin_out.loss, manual_loss, atol=1e-4), "manual shift-by-one loss should match GPT2LMHeadModel's internal computation"
print("manual shift-by-one loss matches the built-in labels= computation: OK")

In [ ]:
prompt_ids = gpt2_tokenizer(causal_text, return_tensors="pt")["input_ids"]

with torch.no_grad():
    greedy_ids = gpt2_model.generate(
        prompt_ids, max_length=prompt_ids.shape[1] + 12, do_sample=False,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
    topk_ids = gpt2_model.generate(
        prompt_ids, max_length=prompt_ids.shape[1] + 12, do_sample=True, top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )

print("prompt:", causal_text)
print("greedy continuation:", gpt2_tokenizer.decode(greedy_ids[0], skip_special_tokens=True))
print("top-k (k=50) sample: ", gpt2_tokenizer.decode(topk_ids[0], skip_special_tokens=True))
if SMOKE_TEST:
    print("SMOKE_TEST=1: gpt2_model has random weights, so the continuations above are gibberish -- expected.")

### In-context learning: 0/1/5-shot toy sentiment classification

GPT-3's headline result was that a frozen, purely autoregressive LM can perform a task from a handful of
input/output examples placed *in the prompt* -- no gradient update, no task-specific head. We demonstrate
the mechanism (not GPT-3's scale) with GPT-2 and a hand-written prompt template:

```
Review: {text}
Sentiment: {label}

Review: {text}
Sentiment: {label}

Review: {query_text}
Sentiment:
```

with 0, 1, or 5 `(text, label)` demonstrations prepended before the query. We then read off the model's
prediction as whichever of `" positive"` / `" negative"` (each a single GPT-2 BPE token) has the higher logit
at the final position -- a deterministic proxy for "did in-context learning teach the model this task,"
avoiding any dependence on free-text generation/parsing.

In [ ]:
# Alternating positive/negative so that any prefix (0, 1, or 5 shots) is reasonably balanced.
few_shot_pool = [
    ("this movie was absolutely wonderful and I loved every minute", "positive"),
    ("what a terrible waste of time, I hated it", "negative"),
    ("the acting was superb and the story kept me engaged", "positive"),
    ("boring plot and terrible dialogue ruined the film", "negative"),
    ("a delightful and heartwarming experience from start to finish", "positive"),
    ("the worst film I have seen all year, deeply disappointing", "negative"),
]

held_out_examples = [
    ("the plot dragged on and the characters were dull", "negative"),
    ("an outstanding performance that left me in tears of joy", "positive"),
    ("I would not recommend this to anyone, complete disappointment", "negative"),
    ("brilliant cinematography and a captivating story from start to end", "positive"),
    ("flat, lifeless, and painfully slow", "negative"),
    ("a joyful, beautifully acted triumph", "positive"),
]

print(f"few-shot pool: {len(few_shot_pool)} examples, held-out set: {len(held_out_examples)} examples")

In [ ]:
def build_few_shot_prompt(query_text, shots):
    '''shots: list of (text, label) demonstration pairs (may be empty for zero-shot).
    query_text: the review text to classify (no label).

    Returns a single string: each shot formatted as "Review: {text}\nSentiment: {label}\n\n", followed by
    "Review: {query_text}\nSentiment:" (no trailing label, no trailing newline -- the model's next token
    IS the prediction).
    '''
    parts = []
    for shot_text, shot_label in shots:
        parts.append(f"Review: {shot_text}\nSentiment: {shot_label}\n\n")
    parts.append(f"Review: {query_text}\nSentiment:")
    return "".join(parts)

In [ ]:
demo_query = held_out_examples[0][0]
for k in [0, 1, 5]:
    demo_prompt = build_few_shot_prompt(demo_query, few_shot_pool[:k])
    print(f"===== {k}-shot prompt =====")
    print(demo_prompt)
    print()

# Verify the prompt-construction + generation code path runs without error, independent of SMOKE_TEST.
five_shot_prompt = build_few_shot_prompt(demo_query, few_shot_pool[:5])
five_shot_ids = gpt2_tokenizer(five_shot_prompt, return_tensors="pt")["input_ids"]
with torch.no_grad():
    five_shot_gen = gpt2_model.generate(
        five_shot_ids, max_length=five_shot_ids.shape[1] + 3, do_sample=False,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
completion = gpt2_tokenizer.decode(five_shot_gen[0, five_shot_ids.shape[1]:], skip_special_tokens=True)
print(f"5-shot prompt -> generated completion: {completion!r}")
print("prompt construction + generation ran without error: OK")

In [ ]:
pos_token_id = gpt2_tokenizer.encode(" positive")[0]
neg_token_id = gpt2_tokenizer.encode(" negative")[0]
assert len(gpt2_tokenizer.encode(" positive")) == 1 and len(gpt2_tokenizer.encode(" negative")) == 1, (
    "expected \" positive\"/\" negative\" to each be a single GPT-2 BPE token"
)


def next_token_sentiment_accuracy(num_shots):
    correct = 0
    for text, true_label in held_out_examples:
        prompt = build_few_shot_prompt(text, few_shot_pool[:num_shots])
        input_ids = gpt2_tokenizer(prompt, return_tensors="pt")["input_ids"]
        with torch.no_grad():
            next_token_logits = gpt2_model(input_ids).logits[0, -1]
        predicted_label = "positive" if next_token_logits[pos_token_id] > next_token_logits[neg_token_id] else "negative"
        correct += int(predicted_label == true_label)
    return correct / len(held_out_examples)


shot_counts = [0, 1, 5]
if SMOKE_TEST:
    print(
        "SMOKE_TEST=1: skipping the few-shot accuracy measurement -- gpt2_model has random weights, so "
        "there is no in-context-learning signal to measure (accuracy would be noise around chance). "
        "Prompt construction and the forward pass were already verified above."
    )
else:
    accuracies = [next_token_sentiment_accuracy(k) for k in shot_counts]
    for k, acc in zip(shot_counts, accuracies):
        print(f"{k}-shot accuracy: {acc:.2f}")

    plt.figure()
    plt.plot(shot_counts, accuracies, marker="o")
    plt.xlabel("number of few-shot demonstrations")
    plt.ylabel("next-token sentiment accuracy")
    plt.title("GPT-2 in-context learning: accuracy vs. number of shots")
    plt.ylim(0, 1.05)
    plt.xticks(shot_counts)
    plt.grid(True, alpha=0.3)
    plt.show()

## Part 3: T5's span-corruption collator

T5's pretraining objective corrupts **contiguous spans** of tokens (not independently-selected individual
tokens -- that would just be MLM again) and replaces each whole span with a single sentinel token
(`<extra_id_0>`, `<extra_id_1>`, ...). The corrupted sequence is the encoder input; the decoder target
concatenates each sentinel with the span of original tokens it replaced, e.g.:

```
input:  the quick brown <extra_id_0> lazy dog near <extra_id_1> afternoon
target: <extra_id_0> fox jumps over the <extra_id_1> a sunny <extra_id_2>
```

We build this via the same randomized-segmentation approach the T5 paper describes: pick how many tokens
total should be corrupted (`noise_density` of the sequence), split that count into `num_spans` contiguous
noise segments (targeting `mean_span_length` tokens each) interleaved with `num_spans` non-noise segments,
using random breakpoints so span lengths and positions vary run to run.

In [ ]:
def _random_segmentation(num_items, num_segments, rng):
    '''Partition num_items into num_segments positive integers (each >= 1) that sum to num_items, by
    choosing num_segments - 1 distinct random breakpoints among the num_items - 1 internal gaps. This is
    NOT the masking decision itself (that's the TODO below) -- it's a helper the masking logic calls to
    decide how long each contiguous span/gap is.
    '''
    assert 1 <= num_segments <= num_items
    breakpoints = sorted(rng.sample(range(1, num_items), num_segments - 1)) if num_segments > 1 else []
    boundaries = [0] + breakpoints + [num_items]
    return [boundaries[i + 1] - boundaries[i] for i in range(num_segments)]


def t5_span_corruption(token_ids, tokenizer, noise_density=0.15, mean_span_length=3, rng=None):
    '''token_ids: list[int], a single example's token ids WITHOUT special tokens.

    Returns (encoder_input_ids, decoder_target_ids), both list[int]:
      - encoder_input_ids: token_ids with each corrupted CONTIGUOUS span replaced by one sentinel token,
        terminated by EOS.
      - decoder_target_ids: for each span in order, its sentinel token followed by the original span
        tokens, followed by one final sentinel and EOS.
    '''
    rng = rng or random
    num_tokens = len(token_ids)
    num_noise_tokens = max(1, min(num_tokens - 1, int(round(num_tokens * noise_density))))
    num_spans = max(1, min(num_noise_tokens, int(round(num_noise_tokens / mean_span_length))))

    noise_lengths = _random_segmentation(num_noise_tokens, num_spans, rng)
    nonnoise_lengths = _random_segmentation(num_tokens - num_noise_tokens, num_spans, rng)

    encoder_input_ids = []
    decoder_target_ids = []
    pos = 0
    for i in range(num_spans):
        # Non-noise (kept) run, then the i-th contiguous noise span, replaced by one sentinel.
        nonnoise_len = nonnoise_lengths[i]
        encoder_input_ids.extend(token_ids[pos:pos + nonnoise_len])
        pos += nonnoise_len

        noise_len = noise_lengths[i]
        span_tokens = token_ids[pos:pos + noise_len]
        pos += noise_len

        sentinel_id = tokenizer.convert_tokens_to_ids(f"<extra_id_{i}>")
        encoder_input_ids.append(sentinel_id)
        decoder_target_ids.append(sentinel_id)
        decoder_target_ids.extend(span_tokens)

    final_sentinel_id = tokenizer.convert_tokens_to_ids(f"<extra_id_{num_spans}>")
    decoder_target_ids.append(final_sentinel_id)
    decoder_target_ids.append(tokenizer.eos_token_id)
    encoder_input_ids.append(tokenizer.eos_token_id)
    return encoder_input_ids, decoder_target_ids

In [ ]:
span_text = (
    "the quick brown fox jumps gracefully over the lazy sleeping dog near the wide flowing river "
    "bank on a sunny afternoon"
)
span_token_ids = t5_tokenizer(span_text, add_special_tokens=False)["input_ids"]

t5_rng = random.Random(0)
encoder_input_ids, decoder_target_ids = t5_span_corruption(
    span_token_ids, t5_tokenizer, noise_density=0.15, mean_span_length=3, rng=t5_rng
)

print("original:      ", " ".join(t5_tokenizer.convert_ids_to_tokens(span_token_ids)))
print("encoder input (corrupted, contiguous spans -> sentinels):")
print(" ", " ".join(t5_tokenizer.convert_ids_to_tokens(encoder_input_ids)))
print("decoder target (sentinel + original span, per span):")
print(" ", " ".join(t5_tokenizer.convert_ids_to_tokens(decoder_target_ids)))

assert len(encoder_input_ids) < len(span_token_ids) + 5, "encoder input should be roughly the original length (spans collapse to 1 sentinel token each)"
assert encoder_input_ids[-1] == t5_tokenizer.eos_token_id
assert decoder_target_ids[-1] == t5_tokenizer.eos_token_id
print("shape/structure checks: OK")

In [ ]:
encoder_input_batch = torch.tensor([encoder_input_ids])
decoder_target_batch = torch.tensor([decoder_target_ids])

with torch.no_grad():
    t5_out = t5_model(input_ids=encoder_input_batch, labels=decoder_target_batch)

print(f"encoder_input shape: {tuple(encoder_input_batch.shape)}, decoder_target shape: {tuple(decoder_target_batch.shape)}")
print(f"logits shape: {tuple(t5_out.logits.shape)} (should match decoder_target's (batch, seq_len))")
assert t5_out.logits.shape[:2] == decoder_target_batch.shape
assert torch.isfinite(t5_out.loss), "T5 span-corruption loss should be finite"
print(f"T5 span-corruption loss: {t5_out.loss.item():.3f}")

if SMOKE_TEST:
    print("SMOKE_TEST=1: t5_model has random weights -- shapes/finiteness verified above; predicted fills below are not meaningful.")
else:
    with torch.no_grad():
        filled_ids = t5_model.generate(encoder_input_batch, max_length=decoder_target_batch.shape[1] + 5)
    print("true fill:      ", t5_tokenizer.decode(decoder_target_batch[0], skip_special_tokens=False))
    print("predicted fill: ", t5_tokenizer.decode(filled_ids[0], skip_special_tokens=False))

## Part 4: attention-pattern visualization -- bidirectional vs. causal

For the same sentence, we extract one layer's, one head's attention-weight matrix from BERT and from GPT-2
(`output_attentions=True`) and plot them side by side. Row `i`, column `j` is how much query position `i`
attends to key position `j`. If BERT is truly bidirectional, its matrix should have non-negligible weight
above the diagonal (position `i` attending to a later position `j > i`); if GPT-2 is truly causal, its
matrix should be exactly lower-triangular (zero everywhere strictly above the diagonal) -- not just
"visually triangular-looking," but exactly zero, which we check programmatically, not just by eye.

In [ ]:
attn_sentence = "the cat sat on the mat"
bert_attn_input = bert_tokenizer(attn_sentence, return_tensors="pt")
gpt2_attn_input = gpt2_tokenizer(attn_sentence, return_tensors="pt")

with torch.no_grad():
    bert_attn_out = bert_model(**bert_attn_input, output_attentions=True)
    gpt2_attn_out = gpt2_model(**gpt2_attn_input, output_attentions=True)

layer_idx, head_idx = 0, 0
bert_attn = bert_attn_out.attentions[layer_idx][0, head_idx]  # (seq_bert, seq_bert)
gpt2_attn = gpt2_attn_out.attentions[layer_idx][0, head_idx]  # (seq_gpt2, seq_gpt2)

bert_tokens = bert_tokenizer.convert_ids_to_tokens(bert_attn_input["input_ids"][0])
gpt2_tokens = gpt2_tokenizer.convert_ids_to_tokens(gpt2_attn_input["input_ids"][0])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, attn, tokens, title in [
    (axes[0], bert_attn, bert_tokens, "BERT (bidirectional)"),
    (axes[1], gpt2_attn, gpt2_tokens, "GPT-2 (causal)"),
]:
    im = ax.imshow(attn.numpy(), cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    ax.set_title(f"{title}\nlayer {layer_idx}, head {head_idx}")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# Programmatic confirmation, not just visual: strictly-upper-triangular weight (query attends to a LATER
# key position) must be non-negligible for BERT and exactly ~0 for GPT-2.
bert_n = bert_attn.shape[0]
bert_upper_mask = torch.triu(torch.ones(bert_n, bert_n), diagonal=1).bool()
bert_upper_weight = bert_attn[bert_upper_mask].sum().item()

gpt2_n = gpt2_attn.shape[0]
gpt2_upper_mask = torch.triu(torch.ones(gpt2_n, gpt2_n), diagonal=1).bool()
gpt2_upper_weight = gpt2_attn[gpt2_upper_mask].abs().max().item()

print(f"BERT: total attention weight strictly above the diagonal = {bert_upper_weight:.3f} (bidirectional -> should be > 0)")
print(f"GPT-2: max attention weight strictly above the diagonal = {gpt2_upper_weight:.2e} (causal -> should be ~0)")
assert bert_upper_weight > 1e-3, "BERT should attend to future positions (bidirectional)"
assert gpt2_upper_weight < 1e-6, "GPT-2 should never attend to future positions (causal)"
print("confirmed: BERT's attention matrix is not lower-triangular, GPT-2's is. OK")

## Summary

| Model | Pretraining objective | Architecture class | Typical downstream use |
|---|---|---|---|
| BERT | Masked language modeling (80/10/10 rule) + (originally) next-sentence prediction | Encoder-only, bidirectional self-attention | Classification, tagging, extractive QA -- tasks needing a full-sequence contextual representation, fine-tuned with a task-specific head |
| GPT-2 | Causal (autoregressive) language modeling: predict token `t+1` from tokens `<=t` | Decoder-only, causal self-attention | Open-ended generation; at sufficient scale (GPT-3), zero/few-shot task transfer via in-context learning, no gradient update |
| T5 | Span corruption: contiguous spans replaced by sentinel tokens, reconstructed by the decoder | Encoder-decoder (encoder bidirectional, decoder causal + cross-attention to the encoder) | Any task reframed as text-to-text (translation, summarization, classification-as-generation) via task prefixes |

**A note on `SMOKE_TEST`**: this notebook runs correctly end-to-end under `SMOKE_TEST=1` (random-weight
models, config-only downloads) -- every masking/collation function, every tensor shape, and the
bidirectional-vs-causal attention-structure check are all verified regardless of weights, since those are
properties of the *code* and the *architecture*, not of what the model has learned. What `SMOKE_TEST` skips
is exactly the handful of cells whose output is only meaningful with real learned weights: the masked-token
top-1 accuracy being above chance, the in-context-learning accuracy-vs-shots curve, and the quality of T5's
predicted span fills. Run this notebook in Colab with `SMOKE_TEST` unset to see those real effects.